In [1]:
import pypdf

# Path to test document inside documents/
pdf_path = "documents/1706.03762v7.pdf"

reader = pypdf.PdfReader(pdf_path)
print(f"Total pages: {len(reader.pages)}")

# Sanity check: preview first page content
first_page_text = reader.pages[0].extract_text()
print(first_page_text[:500])

Total pages: 15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [2]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents_folder = "documents"

# Collect every PDF in the folder automatically
pdf_files = [f for f in os.listdir(documents_folder) if f.endswith(".pdf")]
print(f"Found {len(pdf_files)} PDF files: {pdf_files}")

# Extract full text from each PDF
all_documents = []
for filename in pdf_files:
    filepath = os.path.join(documents_folder, filename)
    reader = pypdf.PdfReader(filepath)

    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"

    all_documents.append({"filename": filename, "text": full_text})
    print(f"{filename}: {len(full_text)} characters extracted")

Found 2 PDF files: ['1706.03762v7.pdf', '_OceanofPDF.com_Raldoron_Revenant_-_Guy_Haley.pdf']
1706.03762v7.pdf: 39511 characters extracted
_OceanofPDF.com_Raldoron_Revenant_-_Guy_Haley.pdf: 324245 characters extracted


In [3]:
# Split settings: chunk_size defines max characters per chunk,
# chunk_overlap keeps context continuity between adjacent chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

all_chunks = []
for doc in all_documents:
    chunks = text_splitter.split_text(doc["text"])
    for chunk in chunks:
        all_chunks.append({"filename": doc["filename"], "text": chunk})

print(f"Total chunks created: {len(all_chunks)}")

# Inspect one chunk to see what it actually looks like
print("\n--- Sample chunk ---")
print(all_chunks[5]["text"])

Total chunks created: 432

--- Sample chunk ---
are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw global dependencies between input and output.
The Transformer allows for significantly more parallelization and can reach a new state of the art in
translation quality after being trained for as little as twelve hours on eight P100 GPUs.
2 Background
The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU
[16], ByteNet [18] and ConvS2S [9], all of which use convolutional neural networks as basic building
block, computing hidden representations in parallel for all input and output positions. In these models,
the number of operations required to relate signals from two arbitrary input or output positions grows
in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes

In [4]:
import chromadb
from chromadb.utils import embedding_functions

# Ollama-backed embedding function: converts text into vectors locally
ollama_ef = embedding_functions.OllamaEmbeddingFunction(
    url="http://localhost:11434/api/embeddings",
    model_name="nomic-embed-text",
)

# Persistent client: data survives between notebook restarts
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="doculocal_rag",
    embedding_function=ollama_ef,
)

print(f"Collection ready. Current items: {collection.count()}")

Collection ready. Current items: 432


In [5]:
# ChromaDB requires unique string IDs for each entry
ids = [f"chunk_{i}" for i in range(len(all_chunks))]
texts = [chunk["text"] for chunk in all_chunks]
metadatas = [{"source": chunk["filename"]} for chunk in all_chunks]

# This triggers embedding generation via Ollama — may take a few minutes
# depending on chunk count and GPU throughput
collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
)

print(f"Collection now contains: {collection.count()} items")

Collection now contains: 432 items
